# Lecture 13: Unconstrained Optimisation

---

```{note}
Lecture 12 introduced the NLP landscape and established that finding a minimum of a smooth function requires the gradient to vanish — $\nabla f(x^*) = \mathbf{0}$ — with the Hessian confirming the character of the stationary point. But that tells us *where* the optimum is, not *how to find it*. This lecture develops the machinery for actually computing the minimum of an unconstrained smooth function. We study three progressively refined algorithms — **Gradient Descent**, **Newton's Method**, and **BFGS (Quasi-Newton)** — and implement all three using Python's `scipy.optimize.minimize`.
```

**Learning Objectives**

By the end of this notebook, you will be able to:
1. Apply first- and second-order optimality conditions to classify stationary points of a smooth unconstrained objective function as local minima, local maxima, or saddle points.
2. Trace the Gradient Descent, Newton's Method, and BFGS algorithms step-by-step, and explain the trade-offs in convergence speed and computational cost."
3. Implement unconstrained minimisation in Python using `scipy.optimize.minimize`, interpret convergence diagnostics, and draw managerial inferences from the optimal solution.

**Prerequisites**: NLP Principles (Lecture 12); calculus (partial derivatives, Taylor expansion); basic linear algebra (matrix inverse, eigenvalues).

**Estimated time**: 90 minutes (including in-class exercises)

---

## Optimality Conditions

### First-Order Necessary Condition

If $x^*$ is a local optimal of a differentiable function $f : \mathbb{R}^n \to \mathbb{R}$, then the **gradient**:

$$\nabla f(x^*) = \mathbf{0}$$

A point satisfying $\nabla f(x^*) = \mathbf{0}$ is called a **stationary point**. This is a necessary condition, not sufficient — a stationary point can be a minimum, a maximum, or a saddle point.

### Second-Order Conditions

The **Hessian** $\nabla^2 f(x^*)$ — the $n \times n$ matrix of second partial derivatives — tells us the curvature of $f$ at the stationary point and resolves the ambiguity:

| Hessian $\nabla^2 f(x^*)$ | All eigenvalues | Character of $x^*$ |
|---------------------------|-----------------|---------------------|
| Positive definite (PD) | $> 0$ | **Local minimum** |
| Negative definite (ND) | $< 0$ | **Local maximum** |
| Indefinite | Mixed | **Saddle point** |

### Example 1

A logistics firm wants to locate a cross-dock hub at $(x, y)$ to minimise total weighted distance to three demand points in the Chennai region:

| Demand point | Location $(a_k, b_k)$ (km, relative to Tambaram) | Weight $w_k$ (tonnes/day) |
|---|---|---|
| Sriperumbudur | $(−30,\; 10)$ | 40 |
| Maraimalai Nagar | $(−10,\; −15)$ | 25 |
| Chengalpattu | $(5,\; −40)$ | 35 |

Using squared Euclidean distance (a convex, differentiable approximation to Euclidean distance):

$$f(x,y) = \sum_{k=1}^{3} w_k\left[(x - a_k)^2 + (y - b_k)^2\right]$$

Gradient:

$$\frac{\partial f}{\partial x} = 2\sum_{k=1}^3 w_k(x - a_k) = 0 \implies x^* = \frac{\sum_k w_k a_k}{\sum_k w_k}$$

$$\frac{\partial f}{\partial y} = 2\sum_{k=1}^3 w_k(y - b_k) = 0 \implies y^* = \frac{\sum_k w_k b_k}{\sum_k w_k}$$

Hessian:

$$\nabla^2 f = 2(\sum_k w_k)\,\mathbf{I}$$

Thus, for the given example,

$$x^* = \frac{40(-30) + 25(-10) + 35(5)}{100} = \frac{-1200 - 250 + 175}{100} = -12.75 \text{ km}$$

$$y^* = \frac{40(10) + 25(-15) + 35(-40)}{100} = \frac{400 - 375 - 1400}{100} = -13.75 \text{ km}$$

$$\nabla^2 f = 200\,\mathbf{I}$$

Hence, $(x^*, y^*) = (-12.75, -13.75)$ is a global minimum — the optimal hub location is approximately 12.75 km west and 13.75 km south of Tambaram.

### Example 2

Recall the BPR (Bureau of Public Roads) travel-time function:

$$t(x) = t_0\left[1 + 0.15\left(\frac{x}{c}\right)^4\right], \qquad x \geq 0$$

A transport planner wants to minimise system travel time $T(x) = x \cdot t(x)$ — total vehicle-hours on the link — over $x \in \mathbb{R}$ (ignoring non-negativity for now). Setting:

$$T(x) = t_0 x + 0.15\, t_0 \frac{x^5}{c^4}$$

$$T'(x) = t_0 + 0.15\, t_0 \frac{5x^4}{c^4} = t_0\left[1 + \frac{0.75 x^4}{c^4}\right]$$

For $x \geq 0$ and $t_0 > 0$: $T'(x) > 0$ always. No stationary point exists in $\mathbb{R}_+$ — the system travel time is strictly increasing, so the unconstrained minimum is at $x = 0$ (no traffic). This example illustrates that not every objective has an interior stationary point. Note, this is the correct mathematical answer but the real-world problem is constrained by travel demand and route choice behavior (Traffic Assignment).


---

## Iterative Descent Framework

To find the point where the gradient vanishes, i.e., $\nabla f(x^*) = \mathbf{0}$, we start from an initial guess $x^{(0)}$, repeatedly computing a direction $d^{(k)}$ that moves downhill and take a step along it:

$$x^{(k+1)} = x^{(k)} + \alpha^{(k)} d^{(k)}$$

until the gradient is (approximately) zero. The three methods in this lecture differ only in how they choose $d^{(k)}$:

| Method | Direction $d^{(k)}$ | Information used | Convergence |
|--------|---------------------|------------------|-------------|
| Gradient Descent | $-\nabla f(x^{(k)})$ | First-order | Linear |
| Newton's Method | $-[\nabla^2 f(x^{(k)})]^{-1}\nabla f(x^{(k)})$ | Second-order | Quadratic |
| BFGS (Quasi-Newton) | $-H_k^{-1}\nabla f(x^{(k)})$, $H_k \approx \nabla^2 f$ | Second-order | SuperLinear |

## 1. Gradient Descent

Gradient Descent is the simplest iterative descent method. It moves in the direction of the steepest descent at the current point — the **negative gradient** — and requires only first-order (gradient) information:

$$d^{(k)} = -\nabla f(x^{(k)})$$

$$x^{(k+1)} = x^{(k)} + \alpha^{(k)} d^{(k)} = x^{(k)} - \alpha^{(k)} \nabla f(x^{(k)})$$

The negative gradient points in the direction of the steepest downhill slope of $f$ at $x^{(k)}$. It is the direction in which $f$ decreases fastest locally — making it a natural, if myopic, choice. However, the gradient carries no information about the curvature of $f$, so the method does not know whether the landscape ahead is flat or sharply curved and must compensate through careful selection of step size $\alpha^{(k)}$ — how far we move along the descent direction. It can be:

- Static: fixed across all iterations; simple, but risks overshooting (too large) or crawling (too small).
- Dynamic: at each iteration, find $\alpha^{(k)} = \arg\min_{\alpha > 0} f(x^{(k)} - \alpha \nabla f(x^{(k)}))$; more robust and used in practice.

Gradient Descent converges *linearly*: the error reduces by a constant factor at each iteration, requiring many steps near the optimum where gradients are small.

## 2. Newton's Method

While Gradient Descent uses only first-order information, Newton's Method incorporates second-order (curvature) information to improve the search direction. It fits a local quadratic model of $f$ around $x^{(k)}$ using the Taylor expansion:

$$f(x^{(k)} + d) \approx f(x^{(k)}) + \nabla f(x^{(k)})^\top d + \frac{1}{2}\, d^\top \nabla^2 f(x^{(k)})\, d$$

Minimising this quadratic approximation over $d$ gives the **Newton direction**:

$$\nabla^2 f(x^{(k)})\, d = -\nabla f(x^{(k)}) \implies d^{(k)} = -[\nabla^2 f(x^{(k)})]^{-1}\nabla f(x^{(k)})$$

$$x^{(k+1)} = x^{(k)} + d^{(k)} = x^{(k)} - [\nabla^2 f(x^{(k)})]^{-1}\nabla f(x^{(k)})$$

The resulting equation can be interpreted as Gradient Descent with step size evaluated via the Hessian $\nabla^2 f(x^{(k)})$ that encodes the curvature of the landscape. Thus, in directions where $f$ curves steeply, the step is short; in directions where $f$ is nearly flat, the step is long. This curvature-aware rescaling allows Newton's Method to take a well-calibrated step towards the optimum in a single move, rather than zig-zagging as Gradient Descent does in ill-conditioned problems.

Newton's Method converges quadratically near the optimum: the number of correct decimal digits roughly doubles at each iteration. However, it requires computing and inverting the full $n \times n$ Hessian at every step — expensive for large $n$.

## 3. Quasi-Newton Method

While the Newton's Method is fast, it demands the full Hessian $\nabla^2 f(x^{(k)})$ at every iteration. This is analytically and numerically expensive for large problems. To this end, Quasi-Newton methods retain the curvature-aware structure of Newton's direction while replacing the true Hessian with a matrix $H_k$ built entirely from first-order information accumulated across iterations.

The Broyden–Fletcher–Goldfarb–Shanno (BFGS) algorithm is the most widely used quasi-Newton method. It initialises $H_0 = \mathbf{I}$ (identity, equivalent to Gradient Descent at the first step) and uses the secant condition — the empirical relationship between step displacement and gradient change — to refine $H_k$ after each iteration. Defining:

$$s^{(k)} = x^{(k+1)} - x^{(k)}, \qquad y^{(k)} = \nabla f(x^{(k+1)}) - \nabla f(x^{(k)})$$

the secant condition requires $H_{k+1}\, s^{(k)} = y^{(k)}$, demanding that the updated approximate Hessian reproduces the observed gradient change over the last step — exactly as the true Hessian would via the mean-value theorem. The BFGS rank-2 update satisfies this condition while keeping $H_{k+1}$ symmetric and positive definite:

$$H_{k+1} = H_k - \frac{H_k s^{(k)} (s^{(k)})^\top H_k}{(s^{(k)})^\top H_k s^{(k)}} + \frac{y^{(k)} (y^{(k)})^\top}{(y^{(k)})^\top s^{(k)}}$$

The BFGS search direction and update rule are then:

$$d^{(k)} = -H_k^{-1}\, \nabla f(x^{(k)})$$

$$x^{(k+1)} = x^{(k)} + \alpha^{(k)} d^{(k)}$$

where $\alpha^{(k)}$ is chosen by a line search satisfying the Wolfe conditions.

The BFGS direction is a curvature-aware descent direction, just like Newton's. The difference is that the curvature information in $H_k$ is learned progressively from past gradient observations rather than computed from scratch at each step. Early in the algorithm, when $H_k \approx \mathbf{I}$, the direction is close to gradient descent. As iterations accumulate, $H_k$ refines its estimate of the Hessian and the direction increasingly resembles the Newton direction — without ever requiring a single second derivative to be computed.

BFGS converges *superlinearly* in practice (faster than linear, approaching quadratic near the optimum), at a fraction of the cost of Newton's Method.

```{note}
In practice: use BFGS as the default for smooth unconstrained problems. Use Newton's Method when the Hessian is cheap to compute (low $n$, or available analytically). Use Gradient Descent only for very large-scale machine learning problems where even storing the Hessian approximation is too expensive (stochastic gradient descent).
```

---

## In-Class Exercises

### Exercise 1

A NHAI traffic engineer models system travel time on two competing highway links connecting Pune and Mumbai. Traffic splits between the two links: let $v_1$ (vehicles/hour) use the Expressway and $v_2$ use the old NH48. The total system travel time (vehicle-hours/hour) is:

$$T(v_1, v_2) = v_1 \cdot t_1(v_1) + v_2 \cdot t_2(v_2)$$

where the BPR travel-time functions are:

$$t_1(v_1) = 90\left[1 + 0.15\left(\frac{v_1}{3{,}000}\right)^4\right], \qquad t_2(v_2) = 150\left[1 + 0.15\left(\frac{v_2}{1{,}500}\right)^4\right]$$

Treat the problem as unconstrained over $(v_1, v_2) \in \mathbb{R}^2$.

1. Find the stationary points analytically and classify them using the Hessian.
2. Implement 5 iterations of Gradient Descent and Newton's Method by hand, starting from $(v_1, v_2) = (1000, 500)$.

### Exercise 2

Delhivery is minimising operational cost at its Chennai cross-dock facility. The daily cost as a function of throughput $\lambda$ (tonnes/day) and staffing level $s$ (workers) is:

$$C(\lambda, s) = 2\lambda^2 - 4\lambda s + 3s^2 - 12\lambda - 6s + 50$$

1. Find the stationary points analytically and classify them using the Hessian.
2. Implement Gradient Descent, Newton's Method, and BFGS in Python, starting from $(0, 0)$.

## Take-Away Exercises

### Exercise 1

BMTC is calibrating a demand-response service in Bengaluru. The system cost depends on fleet size $n$ (vehicles) and headway $h$ (minutes between departures). The operator's total cost model is:

$$C(n, h) = 500n + \frac{12{,}000}{h} + \frac{h^2}{0.04} + 80nh$$

where the terms represent vehicle capital cost, missed-demand penalty, passenger waiting cost, and joint operating cost respectively.

1. Find the stationary points analytically and classify them using the Hessian.
2. Implement 5 iterations of Gradient Descent and Newton's Method by hand, starting from $(n, h) = (10, 10)$.
3. BMTC's budget analyst says: "Adding one more vehicle reduces our waiting-cost penalty." Is this true at the optimum? Compute $\partial C/\partial n$ at $(n^*, h^*)$ to verify.

### Exercise 2

An NHAI corridor study models system travel time across two parallel NH links (Mumbai–Pune Expressway and old NH48) as a function of assigned volumes $(v_1, v_2)$ in thousands of vehicles/hour:

$$T(v_1, v_2) = 3v_1^2 - 2v_1 v_2 + 2v_2^2 - 18v_1 - 12v_2 + 80$$

1. Find the stationary points analytically and classify them using the Hessian.
2. Implement Gradient Descent, Newton's Method, and BFGS in Python, starting from $(0, 0)$.
3. NHAI proposes to restrict the Expressway to $v_1 \leq 4$ thousand veh/hr. At the unconstrained optimum, is this constraint binding? What does this tell you about whether a constrained solve is necessary? (Lecture 15 will handle it formally.)

---

## Further Reading

- Nocedal, J. and Wright, S.J. (2006). *Numerical Optimization* (2nd ed.). Springer — Chapter 2 (fundamentals of unconstrained optimisation), Chapter 3 (line search methods), Chapter 6 (quasi-Newton methods, BFGS).
- Bazaraa, M.S., Sherali, H.D., and Shetty, C.M. (2006). *Nonlinear Programming: Theory and Algorithms* (3rd ed.). Wiley — Chapter 3 (optimality conditions), Chapter 8 (gradient methods), Chapter 9 (Newton and quasi-Newton).
- SciPy documentation: `scipy.optimize.minimize` — [docs.scipy.org](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)
- BPR (1964). *Traffic Assignment Manual*. US Bureau of Public Roads, Washington DC. — Source of the BPR travel-time function used throughout this module.